# 03c Historical behavior features

Optional, for after the session. Does knowing how a SKU has behaved, beyond its sales lags, rolling means, calendar and hierarchy, improve a global LightGBM?

The control is `lgbm_l1_std_categorical` from notebook 03: L1 objective, per-series standardized target, the 02 lags, rolling means and calendar features, and department, class and subclass as categories. Each experiment adds one coherent feature group to it and changes nothing else.

| model | adds |
|---|---|
| `lgbm_l1_cat_distribution` | demand volatility, momentum and zero frequency |
| `lgbm_l1_cat_shape` | the distribution group plus rolling skewness and kurtosis |
| `lgbm_l1_cat_promo_behavior` | promotion frequency, recency, duration and historical response |
| `lgbm_l1_cat_price` | historical relative price, discount depth, price changes and price volatility |
| `lgbm_l1_cat_price_response` | the price group plus an observational price-response estimate |
| `lgbm_l1_cat_static` | origin-level summaries of demand, zeros, promotions and price |
| `lgbm_l1_cat_dept_agg`, `_class_agg`, `_dept_class_agg`, `_subclass_agg` | demand history of the product's department, class or subclass |
| `lgbm_l1_cat_combined` | the groups that helped on the development origins |

Runs in about ten minutes.


In [1]:
import os
import sys
import warnings

if "google.colab" in sys.modules:
    !git clone -q https://github.com/tackes/time-series-forecasting-bootcamp
    %cd time-series-forecasting-bootcamp
    %pip install -q statsforecast==2.1.1 mlforecast==1.1.0 utilsforecast==0.2.17 lightgbm==4.7.0 catboost==1.2.10
elif not os.path.exists("shared"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

In [2]:
import operator
import time

import numpy as np
import pandas as pd
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd, RollingMax, Combine
from mlforecast.target_transforms import LocalStandardScaler

from shared.prepare_data import load_panel, series_segments, H, FREQ, ALL_CUTOFFS, DEV_CUTOFFS, TEST_CUTOFF
from shared.features import LAGS, DATE_FEATURES, LGB_PARAMS
from shared.evaluation import score
from shared.leaderboard import to_long, save_forecasts, load_forecasts, leaderboard, breakdown

panel = load_panel().sort_values(["unique_id", "ds"]).reset_index(drop=True)
panel["promo"] = panel.is_promotional.astype(int)
panel["rel_price"] = (panel.sale_price / panel.regular_price.where(panel.regular_price > 0)).fillna(1).clip(0, 3)
actuals = panel[["unique_id", "ds", "y"]]

## 1. What is known when

The forecast is 14 weeks long, and the features fall into three kinds:

- **Recursive, from the target.** MLForecast lag transforms such as `RollingStd` on lag 1. In the forecast weeks they are recomputed from the model's own forecasts, like the control's lags and rolling means. They are computed on the standardized target, so they are already scale-free.
- **History as of 14 weeks earlier.** For a row in week *t*, a statistic of the weeks up to *t* - 14. For every forecast week that is data up to the cutoff, so it goes in `X_df` with no assumption about the future. Promotion and price features are of this kind, so no forecast depends on promotions inside the horizon.
- **Static per origin.** Summaries of each SKU's training history at the cutoff, recomputed at every origin and constant over the horizon.

`past()` builds the second kind. Every rolling window ends 14 weeks before the row's week, within each series.

In [3]:
def past(values, window, how, min_periods=1):
    """Rolling statistic per series over the `window` weeks ending H weeks before each row's week."""
    return values.groupby(panel.unique_id).transform(lambda s: s.shift(H).rolling(window, min_periods=min_periods).agg(how))


def weeks_since(event):
    """Weeks from the event's last occurrence to H weeks before each row's week (capped at 52)."""
    last = panel.ds.where(event).groupby(panel.unique_id).ffill().groupby(panel.unique_id).shift(H)
    return ((panel.ds - last).dt.days / 7 - H).clip(upper=52).fillna(52)


y = panel.y
features = panel[["unique_id", "ds"]].copy()

## 2. Demand distribution and volatility

A moving average says how much a SKU sells. These say how it sells: how much weeks vary, how often they are zero, and whether recent demand runs above or below the longer average. A global model can use them to tell a stable product from an intermittent or spiky one with the same average.

Recursive: standard deviation over 4, 13 and 26 weeks, the 13-week maximum, and momentum as the 4-week minus the 26-week mean. History as of 14 weeks earlier: zero share over 13 and 26 weeks, coefficient of variation over 26 weeks (at least 8 weeks, positive mean) and weeks since the last sale. Zero share and CV need the original units, which the standardized target no longer has.

The shape extension adds 26-week skewness and kurtosis (at least 8 weeks).

In [4]:
distribution_transforms = {1: [
    RollingMean(window_size=4), RollingMean(window_size=13), RollingMean(window_size=26),
    RollingStd(window_size=4), RollingStd(window_size=13), RollingStd(window_size=26), RollingMax(window_size=13),
    Combine(RollingMean(window_size=4), RollingMean(window_size=26), operator.sub),
]}

is_zero = (y == 0).astype(float)
features["zero_share_13"] = past(is_zero, 13, "mean")
features["zero_share_26"] = past(is_zero, 26, "mean")
features["cv_26"] = past(y, 26, "std", 8) / past(y, 26, "mean", 8).where(lambda m: m > 0)
features["weeks_since_sale"] = weeks_since(y > 0)
features["skew_26"] = past(y, 26, "skew", 8)
features["kurt_26"] = past(y, 26, "kurt", 8)

## 3. Promotion behavior

Frequency, recency and duration come from the promotion flags; response compares sales in promotion and non-promotion weeks. A promotion **event** is a run of consecutive promotion weeks, so a four-week promotion is one event and four promotion weeks.

- Promotion weeks in the last 13 and 26 weeks, events in the last 26, weeks since the last promotion, and the length of the most recent event.
- Over the last 52 weeks: mean sales in promotion weeks over mean sales in other weeks (at least 3 promotion and 8 other weeks), and mean sales in the first week after a promotion over the same reference (at least 2 such weeks). Missing otherwise; LightGBM handles missing values.

All as of 14 weeks earlier. These are associations: promotions are chosen, not assigned at random.

In [5]:
promo = panel.promo.astype(float)
previous = panel.groupby("unique_id").promo.shift(1).fillna(0)
onset = ((panel.promo == 1) & (previous == 0)).astype(float)
after = ((panel.promo == 0) & (previous == 1)).astype(float)
run_length = promo.groupby((panel.promo != panel.groupby("unique_id").promo.shift()).cumsum()).cumsum()

features["promo_weeks_13"] = past(promo, 13, "sum")
features["promo_weeks_26"] = past(promo, 26, "sum")
features["promo_events_26"] = past(onset, 26, "sum")
features["weeks_since_promo"] = weeks_since(panel.promo == 1)
features["last_event_weeks"] = run_length.where(promo == 1).groupby(panel.unique_id).ffill().groupby(panel.unique_id).shift(H).fillna(0)

promo_n, base_n, after_n = past(promo, 52, "sum"), past(1 - promo, 52, "sum"), past(after, 52, "sum")
base_mean = past(y * (1 - promo), 52, "sum") / base_n
features["promo_uplift_52"] = (past(y * promo, 52, "sum") / promo_n / base_mean).where((promo_n >= 3) & (base_n >= 8) & (base_mean > 0)).clip(upper=10)
features["post_promo_ratio_52"] = (past(y * after, 52, "sum") / after_n / base_mean).where((after_n >= 2) & (base_n >= 8) & (base_mean > 0)).clip(upper=10)

## 4. Price history and price response

`rel_price` is the realized average selling price over the regular price. It is partly an outcome: a week with no sales gets the regular price (notebook 03), and 37% of weeks with sales sell above the regular price, which points to mix effects in the average. So the features use relative price only as history:

- Mean relative price over 13 weeks, mean discount depth over 26, the share of weeks in the last 26 where relative price changed by more than 2%, and the 13-week standard deviation.
- **Price response**: the slope of log sales on log relative price over the last 52 weeks, using only weeks with sales. It needs 20 such weeks and some price variation, and is shrunk toward zero by `n var / (n var + 1)`, so a slope from little variation stays close to zero.

The slope is an observational association, not an elasticity. Across SKUs it is negative for 89% (median about -1.9), and estimates from the first and second half of each history correlate at only 0.38.

In [6]:
rel = panel.rel_price
features["rel_price_mean_13"] = past(rel, 13, "mean")
features["discount_depth_26"] = past((1 - rel).clip(lower=0), 26, "mean")
features["price_change_share_26"] = past((rel.groupby(panel.unique_id).diff().abs() > 0.02).astype(float), 26, "mean")
features["price_std_13"] = past(rel, 13, "std", 4)

selling = (y > 0) & (rel > 0)
log_price, log_sales = np.log(rel.where(selling)), np.log(y.where(selling))
n = past(selling.astype(float), 52, "sum")
mean_p, mean_s = past(log_price, 52, "mean"), past(log_sales, 52, "mean")
var_p = past(log_price ** 2, 52, "mean") - mean_p ** 2
cov_ps = past(log_price * log_sales, 52, "mean") - mean_p * mean_s
slope = (cov_ps / var_p).where((n >= 20) & (var_p > 1e-3))
features["price_response_52"] = slope * n * var_p / (n * var_p + 1)

## 5. Static summaries per origin

Similar to the static features in the Day 2 neural models, but recomputed from the training rows at each cutoff, never from the full history. They are constant for a series, so they go in `static_features` and MLForecast repeats them for the forecast weeks.

In [7]:
def origin_summary(history):
    g = history.groupby("unique_id")
    mean = g.y.mean()
    return pd.DataFrame({
        "log_mean_sales": np.log1p(mean),
        "cv_sales": g.y.std() / mean.where(mean > 0),
        "share_zero_weeks": (history.y == 0).groupby(history.unique_id).mean(),
        "share_promo_weeks": g.promo.mean(),
        "promo_events_per_year": onset[history.index].groupby(history.unique_id).sum() / g.size() * 52,
        "rel_price_std": g.rel_price.std(),
    }).reset_index()


static_cols = ["log_mean_sales", "cv_sales", "share_zero_weeks", "share_promo_weeks", "promo_events_per_year", "rel_price_std"]

## 6. Hierarchy demand

The categorical attributes tell the model which group a product belongs to. These describe how that group's demand has moved: weekly sales summed over the products of the 2,000-series sample in the same department, class or subclass. Membership is a fixed item attribute and the sample is fixed, so no future product activity decides who is in a group. A group total grows when a product starts selling, which is why the features are relative:

- Totals 14, 18, 26 and 52 weeks back, and 4-, 13- and 26-week means ending 14 weeks back, each divided by the 52-week mean ending 14 weeks back.
- 13-week coefficient of variation, momentum (4-week over 26-week mean), change from 18 to 14 weeks back, and the group's 13-week promotion share, all ending 14 weeks back.

In [8]:
cat_cols = ["department", "class", "subclass"]
items = panel.drop_duplicates("unique_id").set_index("unique_id")
attributes = pd.DataFrame({col: items[f"{col}_id"].astype("Int64").astype("string").fillna("Unknown") for col in cat_cols}).rename_axis("unique_id").reset_index()
membership = panel[["unique_id", "ds"]].merge(attributes, on="unique_id", how="left")

aggregate_cols = {}
for level in cat_cols:
    weekly = panel.assign(group=membership[level]).groupby(["group", "ds"]).agg(total=("y", "sum"), promo_share=("promo", "mean")).reset_index()
    by_group = weekly.groupby("group")
    lagged = by_group.total.shift(H)
    rolled = lambda values, window, how: values.groupby(weekly.group).transform(lambda s: s.rolling(window, min_periods=1).agg(how))
    reference = lagged.groupby(weekly.group).transform(lambda s: s.rolling(52, min_periods=26).mean())

    group_features = pd.DataFrame({
        "group": weekly.group, "ds": weekly.ds,
        f"{level}_lag14": lagged / reference,
        f"{level}_lag18": by_group.total.shift(18) / reference,
        f"{level}_lag26": by_group.total.shift(26) / reference,
        f"{level}_lag52": by_group.total.shift(52) / reference,
        f"{level}_mean4": rolled(lagged, 4, "mean") / reference,
        f"{level}_mean13": rolled(lagged, 13, "mean") / reference,
        f"{level}_mean26": rolled(lagged, 26, "mean") / reference,
        f"{level}_cv13": rolled(lagged, 13, "std") / rolled(lagged, 13, "mean"),
        f"{level}_momentum": rolled(lagged, 4, "mean") / rolled(lagged, 26, "mean"),
        f"{level}_change": lagged / by_group.total.shift(18) - 1,
        f"{level}_promo_share13": rolled(by_group.promo_share.shift(H), 13, "mean"),
    }).replace([np.inf, -np.inf], np.nan)
    aggregate_cols[level] = list(group_features.columns[2:])
    features = features.assign(group=membership[level].values).merge(group_features, on=["group", "ds"], how="left").drop(columns="group")

features[["unique_id", "ds"] + aggregate_cols["department"][:4]].dropna().head(3)

,unique_id,ds,department_lag14,department_lag18,department_lag26,department_lag52
52,10006_S1,2024-07-03,1.149846,0.907158,1.067984,0.742815
53,10006_S1,2024-07-10,0.895604,1.373186,1.222724,0.935201
54,10006_S1,2024-07-17,0.958766,1.232249,1.006687,0.741571


## 7. The experiments

Each entry lists the dynamic columns (in the training frame and `X_df`), the lag transforms and the static columns. Every model gets the three categorical attributes and the control's L1 LightGBM.

In [9]:
control_transforms = {1: [RollingMean(window_size=4), RollingMean(window_size=13), RollingMean(window_size=26)]}
distribution_cols = ["zero_share_13", "zero_share_26", "cv_26", "weeks_since_sale"]
promo_cols = ["promo_weeks_13", "promo_weeks_26", "promo_events_26", "weeks_since_promo", "last_event_weeks", "promo_uplift_52", "post_promo_ratio_52"]
price_cols = ["rel_price_mean_13", "discount_depth_26", "price_change_share_26", "price_std_13"]

experiments = {
    "lgbm_l1_std_categorical": dict(cols=[], transforms=control_transforms, statics=[]),
    "lgbm_l1_cat_distribution": dict(cols=distribution_cols, transforms=distribution_transforms, statics=[]),
    "lgbm_l1_cat_shape": dict(cols=distribution_cols + ["skew_26", "kurt_26"], transforms=distribution_transforms, statics=[]),
    "lgbm_l1_cat_promo_behavior": dict(cols=promo_cols, transforms=control_transforms, statics=[]),
    "lgbm_l1_cat_price": dict(cols=price_cols, transforms=control_transforms, statics=[]),
    "lgbm_l1_cat_price_response": dict(cols=price_cols + ["price_response_52"], transforms=control_transforms, statics=[]),
    "lgbm_l1_cat_static": dict(cols=[], transforms=control_transforms, statics=static_cols),
    "lgbm_l1_cat_dept_agg": dict(cols=aggregate_cols["department"], transforms=control_transforms, statics=[]),
    "lgbm_l1_cat_class_agg": dict(cols=aggregate_cols["class"], transforms=control_transforms, statics=[]),
    "lgbm_l1_cat_dept_class_agg": dict(cols=aggregate_cols["department"] + aggregate_cols["class"], transforms=control_transforms, statics=[]),
    "lgbm_l1_cat_subclass_agg": dict(cols=aggregate_cols["subclass"], transforms=control_transforms, statics=[]),
}

The training code. The control is refit here only to time it with the same code; its scores use the saved notebook 03 forecasts.

In [10]:
def l1_model(name, lag_transforms):
    return MLForecast(
        models={name: lgb.LGBMRegressor(**dict(LGB_PARAMS, objective="l1"))},
        freq=FREQ,
        lags=LAGS,
        lag_transforms=lag_transforms,
        date_features=DATE_FEATURES,
        target_transforms=[LocalStandardScaler()],
        num_threads=1,
    )


def run_experiments(experiments):
    forecasts, seconds = [], {}
    for cutoff in ALL_CUTOFFS:
        history = actuals[actuals.ds <= cutoff]
        horizon = features[(features.ds > cutoff) & (features.ds <= cutoff + pd.Timedelta(weeks=H))]
        summary = origin_summary(panel[panel.ds <= cutoff])
        for name, spec in experiments.items():
            train = history.merge(features[["unique_id", "ds"] + spec["cols"]], on=["unique_id", "ds"])
            train = train.merge(attributes, on="unique_id").merge(summary[["unique_id"] + spec["statics"]], on="unique_id")
            train = train.astype({col: "category" for col in cat_cols})

            start = time.perf_counter()
            mlf = l1_model(name, spec["transforms"])
            mlf.fit(train, static_features=cat_cols + spec["statics"])
            preds = mlf.predict(H, X_df=horizon[["unique_id", "ds"] + spec["cols"]] if spec["cols"] else None)
            seconds[name] = seconds.get(name, 0) + time.perf_counter() - start
            forecasts.append(to_long(preds, cutoff, "03c_behavior_features"))
    return pd.concat(forecasts), seconds, mlf


behavior_forecasts, fit_seconds, last_model = run_experiments(experiments)

The attributes are still treated as categories, and the new columns as numbers (last model fitted, final origin):

In [11]:
infos = last_model.models_["lgbm_l1_cat_subclass_agg"].booster_.dump_model()["feature_infos"]
pd.Series({f: "categorical" if i["values"] else "numeric" for f, i in infos.items()}).value_counts()

numeric        25
categorical     3
Name: count, dtype: int64

## 8. Results

Each model against the control on identical rows. Fit time is fit plus 14-week forecast, summed over the ten origins.

In [12]:
control = load_forecasts(["03_categorical"]).query("model == 'lgbm_l1_std_categorical'")
new_models = [name for name in experiments if name != "lgbm_l1_std_categorical"]
candidates = pd.concat([control, behavior_forecasts[behavior_forecasts.model.isin(new_models)]])

columns = ["model", "wape", "bias", "mean_rmsse", "median_rmsse", "apc", "coverage"]
dev_board = leaderboard(candidates, actuals, scope="dev").set_index("model")
test_board = leaderboard(candidates, actuals, scope="test").set_index("model")
table = pd.concat({"development": dev_board[columns[1:]], "final 14 weeks": test_board[columns[1:3]]}, axis=1)
table[("fit", "seconds")] = pd.Series(fit_seconds)
table.sort_values(("development", "wape")).round(3)

development                                        \
                                  wape   bias mean_rmsse median_rmsse    apc   
model                                                                          
lgbm_l1_cat_price_response       0.353 -0.085      1.719        0.831  0.136   
lgbm_l1_cat_price                0.354 -0.077      1.721        0.832  0.136   
lgbm_l1_cat_static               0.357 -0.078      1.722        0.826  0.139   
lgbm_l1_cat_subclass_agg         0.357 -0.060      1.712        0.831  0.139   
lgbm_l1_cat_promo_behavior       0.358 -0.079      1.722        0.832  0.137   
lgbm_l1_cat_dept_class_agg       0.358 -0.059      1.712        0.829  0.141   
lgbm_l1_std_categorical          0.359 -0.072      1.728        0.830  0.140   
lgbm_l1_cat_class_agg            0.359 -0.059      1.715        0.829  0.140   
lgbm_l1_cat_dept_agg             0.359 -0.062      1.723        0.832  0.142   
lgbm_l1_cat_shape                0.360 -0.065      1.727        0.835  0.143   
lgbm_l1_cat_distribution         0.362 -0.067      1.720        0.835  0.141   

                                    final 14 weeks            fit  
                           coverage           wape   bias seconds  
model                                                              
lgbm_l1_cat_price_response      1.0          0.344 -0.035  11.136  
lgbm_l1_cat_price               1.0          0.348 -0.029  10.424  
lgbm_l1_cat_static              1.0          0.361 -0.017  11.403  
lgbm_l1_cat_subclass_agg        1.0          0.346 -0.005  11.886  
lgbm_l1_cat_promo_behavior      1.0          0.348 -0.028  11.276  
lgbm_l1_cat_dept_class_agg      1.0          0.345 -0.008  12.605  
lgbm_l1_std_categorical         1.0          0.350 -0.020  10.829  
lgbm_l1_cat_class_agg           1.0          0.345 -0.012  11.424  
lgbm_l1_cat_dept_agg            1.0          0.348 -0.013  11.612  
lgbm_l1_cat_shape               1.0          0.354 -0.023  11.816  
lgbm_l1_cat_distribution        1.0          0.354 -0.024  12.119

Change in WAPE points against the control at every origin; `beats_dev` counts the seven development origins where the model is better.

In [13]:
origins = list(DEV_CUTOFFS) + [TEST_CUTOFF]
by_origin = pd.concat([score(candidates[candidates.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in origins])
origin_wape = by_origin.pivot(index="model", columns="cutoff", values="wape")
vs_control = (origin_wape - origin_wape.loc["lgbm_l1_std_categorical"]) * 100
vs_control.drop(index="lgbm_l1_std_categorical").round(2).assign(beats_dev=(vs_control[list(DEV_CUTOFFS)] < 0).sum(axis=1))

cutoff,2024-07-17 00:00:00,2024-08-14 00:00:00,2024-09-11 00:00:00,2024-10-09 00:00:00,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-04-09 00:00:00,beats_dev
model,,,,,,,,,
lgbm_l1_cat_class_agg,-0.02,0.07,-0.50,-0.15,-0.12,0.59,0.54,-0.60,4
lgbm_l1_cat_dept_agg,0.10,-0.49,-0.38,-0.12,0.36,0.42,0.62,-0.21,3
lgbm_l1_cat_dept_class_agg,0.40,-0.49,-0.74,-0.31,-0.15,0.55,0.54,-0.55,4
lgbm_l1_cat_distribution,0.27,0.62,-0.22,0.84,0.08,0.39,0.08,0.36,1
lgbm_l1_cat_price,-1.19,-0.87,-1.05,-0.19,0.14,-0.26,-0.12,-0.22,6
lgbm_l1_cat_price_response,-1.12,-1.09,-0.85,-0.25,-0.13,-0.39,-0.32,-0.66,7
lgbm_l1_cat_promo_behavior,-0.26,-0.46,0.13,0.00,-0.12,0.06,0.04,-0.20,3
lgbm_l1_cat_shape,1.27,-0.35,-0.77,0.79,-0.17,0.35,-0.35,0.37,4
lgbm_l1_cat_static,-0.69,-0.31,-0.40,0.65,0.10,0.17,-1.09,1.02,4


### By product characteristics

Segments are recomputed at each origin from history up to that cutoff: volume and intermittency with `series_segments(panel, cutoff)`, and promotion frequency as the share of promotion weeks (rarely up to 5%, occasionally up to 25%, heavily above). Development origins; WAPE change in points against the control.

In [14]:
def origin_segments(cutoff):
    segments = series_segments(panel, cutoff)
    share = panel[panel.ds <= cutoff].groupby("unique_id").promo.mean()
    segments["promotion"] = pd.cut(share, [-1, 0.05, 0.25, 1], labels=["rarely", "occasionally", "heavily"]).astype(str)
    return segments.assign(cutoff=cutoff).rename_axis("unique_id").reset_index()


segment_rows = candidates[candidates.cutoff.isin(DEV_CUTOFFS)].merge(pd.concat([origin_segments(c) for c in DEV_CUTOFFS]), on=["unique_id", "cutoff"])
segment_change = {}
for segment, order in [("volume", ["low", "mid", "high"]), ("intermittency", ["regular", "some_zeros", "zero_heavy"]), ("promotion", ["rarely", "occasionally", "heavily"])]:
    wape = score(segment_rows, actuals, by=[segment], rmsse=False).pivot(index="model", columns=segment, values="wape")[order]
    segment_change[segment] = (wape - wape.loc["lgbm_l1_std_categorical"]) * 100
pd.concat(segment_change, axis=1).drop(index="lgbm_l1_std_categorical").round(2)

volume             intermittency             \
                              low   mid  high       regular some_zeros   
model                                                                    
lgbm_l1_cat_class_agg       -0.12 -0.02  0.06          0.02      -0.07   
lgbm_l1_cat_dept_agg        -0.21 -0.00  0.09          0.20      -0.25   
lgbm_l1_cat_dept_class_agg  -0.02 -0.07 -0.03         -0.08      -0.23   
lgbm_l1_cat_distribution     0.15 -0.01  0.32          0.23       0.08   
lgbm_l1_cat_price            0.20 -0.06 -0.53         -0.47      -0.58   
lgbm_l1_cat_price_response   0.01 -0.18 -0.62         -0.45      -0.85   
lgbm_l1_cat_promo_behavior  -0.16 -0.12 -0.08         -0.08      -0.09   
lgbm_l1_cat_shape            0.21  0.09  0.10          0.41      -0.61   
lgbm_l1_cat_static          -2.04 -0.99 -0.11         -0.18      -0.59   
lgbm_l1_cat_subclass_agg    -0.20  0.03 -0.14         -0.04      -0.33   

                                      promotion                       
                           zero_heavy    rarely occasionally heavily  
model                                                                 
lgbm_l1_cat_class_agg            2.78      0.09         0.13    0.02  
lgbm_l1_cat_dept_agg            -0.09      0.11         0.01    0.09  
lgbm_l1_cat_dept_class_agg       3.95      0.02         0.01   -0.07  
lgbm_l1_cat_distribution         4.94      0.02         0.82    0.19  
lgbm_l1_cat_price                0.02     -0.18        -0.29   -0.66  
lgbm_l1_cat_price_response      -1.75     -0.07        -0.57   -0.75  
lgbm_l1_cat_promo_behavior       0.08     -0.07        -0.12   -0.07  
lgbm_l1_cat_shape               -2.39      0.16         0.25    0.02  
lgbm_l1_cat_static               3.47     -0.26         0.47   -0.44  
lgbm_l1_cat_subclass_agg        -0.63      0.12        -0.35   -0.12

## 9. Combined model, chosen on development evidence

A group goes into the combined model if it lowers development WAPE by at least 0.2 points and beats the control at five or more of the seven development origins. Within a family of overlapping variants (distribution and shape; price and price response; the four aggregate variants) only the one with the lowest development WAPE is kept. The final window plays no part in the choice.

In [15]:
qualifies = ((dev_board.wape["lgbm_l1_std_categorical"] - dev_board.wape) >= 0.002) & (vs_control[list(DEV_CUTOFFS)].lt(0).sum(axis=1) >= 5)
families = [["distribution", "shape"], ["promo_behavior"], ["price", "price_response"], ["static"], ["dept_agg", "class_agg", "dept_class_agg", "subclass_agg"]]
chosen = []
for family in families:
    passing = [f"lgbm_l1_cat_{group}" for group in family if qualifies[f"lgbm_l1_cat_{group}"]]
    chosen += [min(passing, key=dev_board.wape.get)] if passing else []
chosen

['lgbm_l1_cat_price_response']

In [16]:
combined = dict(
    cols=sorted({col for name in chosen for col in experiments[name]["cols"]}),
    transforms=distribution_transforms if {"lgbm_l1_cat_distribution", "lgbm_l1_cat_shape"} & set(chosen) else control_transforms,
    statics=sorted({col for name in chosen for col in experiments[name]["statics"]}),
)
combined_forecasts, combined_seconds, _ = run_experiments({"lgbm_l1_cat_combined": combined})

candidates = pd.concat([candidates, combined_forecasts])
fit_seconds |= combined_seconds
best_group = dev_board.drop(index="lgbm_l1_std_categorical").wape.idxmin()
compare = candidates[candidates.model.isin(["lgbm_l1_std_categorical", best_group, "lgbm_l1_cat_combined"])]
pd.concat({"development": leaderboard(compare, actuals, scope="dev").set_index("model")[columns[1:]],
           "final 14 weeks": leaderboard(compare, actuals, scope="test").set_index("model")[columns[1:3]]}, axis=1).round(3)

development                                        \
                                  wape   bias mean_rmsse median_rmsse    apc   
model                                                                          
lgbm_l1_cat_price_response       0.353 -0.085      1.719        0.831  0.137   
lgbm_l1_cat_combined             0.354 -0.083      1.718        0.831  0.136   
lgbm_l1_std_categorical          0.359 -0.072      1.728        0.830  0.141   

                                    final 14 weeks         
                           coverage           wape   bias  
model                                                      
lgbm_l1_cat_price_response      1.0          0.344 -0.035  
lgbm_l1_cat_combined            1.0          0.345 -0.032  
lgbm_l1_std_categorical         1.0          0.350 -0.020

### By horizon

Development WAPE by forecast week for the control, the best single group and the combined model.

In [17]:
breakdown(compare, actuals, "horizon", scope="dev").pivot(index="horizon", columns="model", values="wape").round(3)

model,lgbm_l1_cat_combined,lgbm_l1_cat_price_response,lgbm_l1_std_categorical
horizon,,,
1,0.271,0.271,0.275
2,0.307,0.308,0.313
3,0.312,0.311,0.315
4,0.319,0.319,0.324
5,0.330,0.330,0.334
6,0.367,0.367,0.370
7,0.355,0.353,0.360
8,0.356,0.354,0.361
9,0.360,0.358,0.365


## 10. Reading the result

Control: development WAPE 0.359, final window 0.350.

- **Price history is the only group that helps consistently.** `lgbm_l1_cat_price_response` lowers development WAPE by 0.6 points and beats the control at all seven development origins; `lgbm_l1_cat_price` by 0.5 points at six of seven. The final window agrees (0.344 and 0.348). The gain is spread over every horizon and is largest for high-volume and heavily promoted products. The price-response estimate adds about 0.1 points on development over the price group alone.
- **Promotion behavior adds little.** 0.1 points on development, better at three of seven origins. The promotion flags and calendar already in the history carry most of it; the price features also overlap with promotion frequency (Spearman correlation about 0.7 for discount depth).
- **Hierarchy demand is neutral on development.** Between +0.0 and -0.2 points, better at three or four of seven origins. All four variants look better in the final window (0.345 to 0.348) and move the bias toward zero, but development does not confirm it, so this is not evidence of a gain.
- **Static summaries are unstable.** 0.2 points better on development (four of seven), 1.1 points worse in the final window. They help low-volume products and hurt zero-heavy ones.
- **Distribution, volatility and shape features don't help.** The distribution group is 0.3 points worse on development (better at one origin of seven); skewness and kurtosis don't change that.
- **RMSSE and APC barely move** for any group (mean RMSSE 1.71 to 1.73 against 1.73; APC 0.136 to 0.143 against 0.140). Zero-heavy segments swing by several points either way, because their WAPE rests on very few sales.
- **Cost.** Each model fits in about the same time as the control (10 to 13 seconds over ten origins against 11); building the features takes about a minute once.

Only the price-response group passed the development rule, so the combined model has the same features as `lgbm_l1_cat_price_response`, in a different column order. The two differ by 0.1 points. That is the noise from column subsampling, and it is about the size of most of the differences above.

The price features are historical relative prices, not future ones, and the scrambling test confirms that no future sales, prices or promotions reach these forecasts. Relative price is still partly an outcome of past sales (notebook 03), so read the price-response slope as a description of past co-movement, not an elasticity.

In [18]:
save_forecasts(candidates[candidates.model != "lgbm_l1_std_categorical"], "03c_behavior_features")